# 03 · Limpeza e estruturação dos precedentes

Este notebook transforma os blocos brutos da auditoria num **JSONL único**, `data/processed/precedentes/precedentes_tst.jsonl`, com um registro por súmula, OJ ou tema. Esse arquivo é a entrada do chunking (notebook 04).

**Regras de limpeza (definidas a partir da EDA):**

| Problema no bruto | Regra |
|---|---|
| Linha quebrada e hifenização no fim da linha | Juntar as linhas. "jorna-\nda" vira "jornada", e "ex-\nSúmula" mantém o hífen |
| Cabeçalho e rodapé de página ("SÚMULAS - TST", "SUMÁRIO") | Removidos já na extração |
| Parêntese no título, como "(ART. 612 DA CLT)", e parêntese de situação, como "(cancelada)" | O parêntese em maiúsculas é parte do título. Os símbolos "º" e "ª" não contam como minúscula. Os parênteses de situação são lidos em sequência, porque às vezes há dois seguidos, como "(nova redação) ... (cancelada)", e o último é o que vale. Um parêntese só com a publicação, como "(DEJT ...)", não é situação |
| Publicação ("- Res. 129/2005, DJ 20, 22 e 25.04.2005") | Retirada do enunciado |
| Histórico de redações | Campo `historico`, fora do texto de busca |
| Notas remissivas ao TRT-3 ("• SÚM. 29 TRT3") | Campo `remissoes_trt3` |
| Origem do item ("(ex-OJ nº 234 da SBDI-1...)") | Mantida em `texto` (oficial) e retirada de `texto_busca` |
| Itens I, II, III... (também no formato "I.", como na Súmula 85) | Lista `itens` |
| Tema repetido em vários assuntos | Um registro por tema, com os `assuntos` e as `paginas` reunidos |
| Número da página colado na tese ("Ainda não fixada. 21") | Corrigido |
| Faixa "IRR com determinação de suspensão" | Campo `avisos` |

**Vigência:**

| Situação | Quando | Entra na busca? |
|---|---|---|
| `vigente` | Súmula ou OJ sem cancelamento no cabeçalho, inclusive quando só um item foi cancelado | sim |
| `cancelada` | O cabeçalho diz cancelada ou cancelamento, ou o verbete não tem enunciado porque foi convertido em outro | não, mas fica registrada |
| `suspensa` | Súmula com eficácia ou aplicação suspensa por decisão do STF, como as Súmulas 228 e 277 | não, mas fica registrada |
| `tese_fixada` | Tema com tese no texto | sim |
| `afetado_sem_tese` | Tese "Ainda não fixada". Útil para responder que a questão está pendente no TST | sim |
| `tese_superada` | Tese "Superada" | não |
| alerta da notícia | Cancelamento noticiado sem fonte oficial. A situação oficial não muda | integral: **não** (quarentena); parcial: sim, com o alerta |

### Dados desta etapa

| | Caminho no repositório | Formato | Quantidade | Versionado? |
|---|---|---|---|---|
| Entrada | `data/interim/precedentes/verbetes_brutos.jsonl` | JSONL (um registro JSON por linha) | 1.596 blocos brutos | fora do Git |
| **Saída: a base de precedentes** | `data/processed/precedentes/precedentes_tst.jsonl` | JSONL (um registro JSON por linha) | **1.493 registros** (463 súmulas, 709 OJs, 321 temas), dos quais **1.022 buscáveis**; 23 campos por registro | fora do Git |
| Relatórios | `reports/precedentes/estruturacao/` | CSV e JSON | contagens.csv (8 linhas: tipo × situação), manifesto_estruturacao.json (hash do JSONL) | sim |

**Formato:** a base estruturada é gravada em **JSONL** (texto, um registro JSON por linha), e não em CSV, porque cada registro tem campos com listas (itens da súmula, páginas, assuntos, histórico). Os arquivos **CSV** do projeto são os relatórios em `reports/precedentes/`. Os arquivos de dados ficam em `data/`, dentro da pasta do repositório, mas **fora do Git** (`.gitignore`): contêm texto integral e são recriados pelos notebooks.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/precedentes').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', None)
RAW = ROOT / 'data/raw/precedentes'
BRUTOS = ROOT / 'data/interim/precedentes/verbetes_brutos.jsonl'
SAIDA_JSONL = ROOT / 'data/processed/precedentes/precedentes_tst.jsonl'
REL = ROOT / 'reports/precedentes'

from src.precedentes import documentos, extracao, estruturacao as est
brutos = [json.loads(l) for l in BRUTOS.read_text(encoding='utf-8').splitlines()]
manifesto = json.loads((REL / 'auditoria/manifesto_auditoria.json').read_text(encoding='utf-8'))
inv = {c['sigla']: c for c in documentos.CATALOGO}
def fonte(sigla):
    return dict(sigla=sigla, arquivo=inv[sigla]['arquivo'], sha256=manifesto['fontes'][sigla]['sha256'],
                data_referencia=inv[sigla]['data_referencia'])
alertas = {a['codigo']: a for a in extracao.cancelamentos_noticia(RAW / 'cancelamento_36_enunciados_tst.pdf')}
(REL / 'estruturacao').mkdir(parents=True, exist_ok=True)
print(f"{len(brutos)} blocos brutos · alertas da notícia: {len(alertas)}")

1596 blocos brutos · alertas da notícia: 26


## 1. Estruturação

In [2]:
verbetes = [est.estruturar_verbete(b, fonte('TRT3_2024'), alertas) for b in brutos if b['fonte'] == 'TRT3_2024']
temas = est.estruturar_temas([b for b in brutos if b['fonte'] == 'IRR_2026'], fonte('IRR_2026'))
registros = verbetes + temas
df = pd.DataFrame(registros)
print(f'{len(verbetes)} súmulas e OJs + {len(temas)} temas = {len(registros)} registros')
resumo = pd.crosstab(df.tipo, df.situacao, margins=True, margins_name='total')
display(resumo)
print('Entram na busca:', df.buscavel.groupby(df.tipo).sum().to_dict(), '· total', int(df.buscavel.sum()))

1172 súmulas e OJs + 321 temas = 1493 registros
Entram na busca: {'orientacao_jurisprudencial': 396, 'sumula': 307, 'tema_repetitivo': 319} · total 1022


situacao,afetado_sem_tese,cancelada,suspensa,tese_fixada,tese_superada,vigente,total
tipo,,,,,,,
orientacao_jurisprudencial,0,308,0,0,0,401,709
sumula,0,138,2,0,0,323,463
tema_repetitivo,98,0,0,221,2,0,321
total,98,446,2,221,2,724,1493


## 2. Antes e depois: exemplos

Mostramos um exemplo de cada caso:

| Registro | Caso |
|---|---|
| Súmula 338 | Enunciado com itens |
| OJ 394 da SBDI-1 | Parêntese só com a publicação |
| Súmula 221 | Cancelamento de um item só, então continua vigente |
| OJ 27 da SBDI-1 | Convertida em súmula, então fica cancelada |
| OJ 21 da SDC | Título com parêntese em maiúsculas |
| Tema 128 | Tese fixada |
| Tema 43 | Afetado, ainda sem tese, com aviso de suspensão |

In [3]:
por_id = {r['id']: r for r in registros}
bruto_por_codigo = {b['codigo']: b for b in brutos if b['fonte'] == 'TRT3_2024'}
print('ANTES (bruto) - Súmula 338:\n')
print(bruto_por_codigo['SUM-338']['texto_bruto'][:900], '\n')
print('DEPOIS (campos):')
r = por_id['TST-SUM-338']
for campo in ('nome_citacao', 'titulo', 'situacao', 'situacao_detalhe', 'buscavel'):
    print(f'  {campo}: {r[campo]}')
print('  itens:', [i['item'] for i in r['itens']])
print('  texto_busca:', r['texto_busca'][:400], '…')
print('  historico:', (r['historico'] or '')[:160], '…')
print('  remissoes_trt3:', r['remissoes_trt3'])
cols = ['id', 'titulo', 'situacao', 'situacao_detalhe', 'buscavel', 'texto_busca']
display(pd.DataFrame([por_id[i] for i in ['TST-OJ-SDI1-394', 'TST-SUM-221', 'TST-OJ-SDI1-27', 'TST-OJ-SDC-21',
                                          'TST-IRR-128', 'TST-IRR-43']])[cols + ['avisos']])

ANTES (bruto) - Súmula 338:

SUM-338 JORNADA DE TRABALHO. REGISTRO. ÔNUS DA PROVA (incorporadas as
Orientações Jurisprudenciais nºs 234 e 306 da SBDI-1) - Res. 129/2005, DJ 20, 22 e
25.04.2005
I - É ônus do empregador que conta com mais de 10 (dez) empregados o registro da jornada
de trabalho na forma do art. 74, § 2º, da CLT. A não-apresentação injustificada dos controles
de frequência gera presunção relativa de veracidade da jornada de trabalho, a qual pode ser
elidida por prova em contrário. (ex-Súmula nº 338 – alterada pela Res. 121/2003, DJ
21.11.2003)
II - A presunção de veracidade da jornada de trabalho, ainda que prevista em instrumento
normativo, pode ser elidida por prova em contrário. (ex-OJ nº 234 da SBDI-1 - inserida em
20.06.2001)
III - Os cartões de ponto que demonstram horários de entrada e saída uniformes são inválidos
como meio de prova, invertendo-se o ônus da prova, relativo às horas extras, que  

DEPOIS (campos):
  nome_citacao: Súmula 338 do TST
  titulo: JORNADA

,id,titulo,situacao,situacao_detalhe,buscavel,texto_busca,avisos
0,TST-OJ-SDI1-394,"REPOUSO SEMANAL REMUNERADO - RSR. INTEGRAÇÃO DAS HORAS EXTRAS. NÃO REPERCUSSÃO NO CÁLCULO DAS FÉRIAS, DO DÉCIMO TERC...",vigente,NaN,True,"A majoração do valor do repouso semanal remunerado, em razão da integração das horas extras habitualmente prestadas,...",[]
1,TST-SUM-221,RECURSO DE REVISTA. VIOLAÇÃO DE LEI. INDICAÇÃO DE PRECEITO,vigente,cancelado o item II e conferida nova redação na sessão do Tribunal Pleno realizada em 14.09.2012,True,A admissibilidade do recurso de revista por violação tem como pressuposto a indicação expressa do dispositivo de lei...,[]
2,TST-OJ-SDI1-27,CONAB. ESTABILIDADE CONCEDIDA POR NORMA INTERNA. NÃO ASSEGURADA. AVISO DIREH Nº 02/1984,cancelada,cancelada em decorrência da sua conversão na Súmula nº 355,False,,[]
3,TST-OJ-SDC-21,"ILEGITIMIDADE ""AD CAUSAM"" DO SINDICATO. AUSÊNCIA DE INDICAÇÃO DO TOTAL DE ASSOCIADOS DA ENTIDADE SINDICAL. INSUFICIÊ...",cancelada,cancelada,False,,[]
4,TST-IRR-128,ACÚMULO DE FUNÇÕES. EXERCÍCIO CONCOMITANTE DA FUNÇÃO DE COBRADOR E MOTORISTA DE ÔNIBUS URBANO,tese_fixada,NaN,True,Questão jurídica: O exercício cumulativo das funções de motorista de ônibus urbano e cobrador caracteriza acúmulo de...,[]
5,TST-IRR-43,ADICIONAL DE INSALUBRIDADE. ENQUADRAMENTO. DIFERENÇAS. NORMA COLETIVA,afetado_sem_tese,NaN,True,Questão jurídica: É válida norma coletiva que dispõe sobre o enquadramento do grau de insalubridade para pagamento d...,[IRR com determinação de suspensão no TST – Recursos de Revista e Embargos.]


## 3. Vigência: quarentena dos cancelamentos noticiados

Os enunciados citados na notícia continuam com a situação oficial da coletânea e recebem o alerta. Os cancelados **por inteiro** saem da busca. Os cancelados **em parte** continuam na busca com o alerta, e o modelo deve avisar que aquele item pode não valer mais.

In [4]:
alert = df[df.alerta_vigencia.notna()][['id', 'nome_citacao', 'titulo', 'situacao', 'buscavel', 'alerta_vigencia']]
print(f'{len(alert)} com alerta · em quarentena (fora da busca): {(~alert.buscavel & (alert.situacao == "vigente")).sum()}')
display(alert)

25 com alerta · em quarentena (fora da busca): 21


,id,nome_citacao,titulo,situacao,buscavel,alerta_vigencia
5,TST-SUM-6,Súmula 6 do TST,EQUIPARAÇÃO SALARIAL. ART. 461 DA CLT,vigente,True,"cancelamento noticiado (fonte não oficial; alcance: itens I, II, VI (alínea “b”) e X.)"
89,TST-SUM-90,Súmula 90 do TST,"HORAS ""IN ITINERE"". TEMPO DE SERVIÇO",vigente,False,cancelamento noticiado (fonte não oficial; alcance: integral)
113,TST-SUM-114,Súmula 114 do TST,PRESCRIÇÃO INTERCORRENTE,vigente,False,cancelamento noticiado (fonte não oficial; alcance: integral)
151,TST-SUM-152,Súmula 152 do TST,GRATIFICAÇÃO. AJUSTE TÁCITO,vigente,False,cancelamento noticiado (fonte não oficial; alcance: integral)
267,TST-SUM-268,Súmula 268 do TST,PRESCRIÇÃO. INTERRUPÇÃO. AÇÃO TRABALHISTA ARQUIVADA,vigente,False,cancelamento noticiado (fonte não oficial; alcance: integral)
276,TST-SUM-277,Súmula 277 do TST,CONVENÇÃO COLETIVA DE TRABALHO OU ACORDO COLETIVO DE TRABALHO. EFICÁCIA. ULTRATIVIDADE,suspensa,False,cancelamento noticiado (fonte não oficial; alcance: integral)
293,TST-SUM-294,Súmula 294 do TST,PRESCRIÇÃO. ALTERAÇÃO CONTRATUAL. TRABALHADOR URBANO,vigente,False,cancelamento noticiado (fonte não oficial; alcance: integral)
319,TST-SUM-320,Súmula 320 do TST,"HORAS ""IN ITINERE"". OBRIGATORIEDADE DE CÔMPUTO NA JORNADA DE TRABALHO",vigente,False,cancelamento noticiado (fonte não oficial; alcance: integral)
330,TST-SUM-331,Súmula 331 do TST,CONTRATO DE PRESTAÇÃO DE SERVIÇOS. LEGALIDADE,vigente,True,cancelamento noticiado (fonte não oficial; alcance: Item I)
365,TST-SUM-366,Súmula 366 do TST,CARTÃO DE PONTO. REGISTRO. HORAS EXTRAS. MINUTOS QUE ANTECEDEM E SUCEDEM A JORNADA DE TRABALHO,vigente,False,cancelamento noticiado (fonte não oficial; alcance: integral)


## 4. Validação e gravação do JSONL único

O arquivo só é gravado se passar nas verificações abaixo:
- nenhum `id` repetido;
- nenhum campo obrigatório vazio (tipo, número, nome de citação, situação e origem);
- nenhum registro buscável sem texto de busca.

O manifesto registra o hash do JSONL, das fontes e as contagens. Com isso, o chunking do notebook 04 pode conferir que usa exatamente esta versão.

> **Grava:** `data/processed/precedentes/precedentes_tst.jsonl` (JSONL, 1.493 linhas, fora do Git), `reports/precedentes/estruturacao/contagens.csv` (CSV, 8 linhas) e `manifesto_estruturacao.json`

In [5]:
problemas = est.validar(registros)
print('Problemas:', problemas or 'nenhum')
sha_jsonl = est.gravar_jsonl(registros, SAIDA_JSONL)
contagens = df.groupby(['tipo', 'situacao']).size().rename('registros').reset_index()
contagens.to_csv(REL / 'estruturacao/contagens.csv', index=False, encoding='utf-8')
manifesto_est = dict(arquivo=str(SAIDA_JSONL.relative_to(ROOT)).replace('\\', '/'), sha256=sha_jsonl,
                     registros=len(registros), buscaveis=int(df.buscavel.sum()), esquema=est.ESQUEMA,
                     fontes={s: fonte(s) for s in ('TRT3_2024', 'IRR_2026')},
                     alertas_noticia=len(alert), brutos_sha256=manifesto['brutos_sha256'])
(REL / 'estruturacao/manifesto_estruturacao.json').write_text(json.dumps(manifesto_est, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Gravado: {manifesto_est["arquivo"]} · {len(registros)} registros · {manifesto_est["buscaveis"]} buscáveis · sha256 {sha_jsonl[:16]}…')
display(contagens)
print('\nUm registro do JSONL (Súmula 60):')
print(json.dumps({k: v for k, v in por_id['TST-SUM-60'].items() if k not in ('historico',)}, ensure_ascii=False, indent=1)[:1800])

Problemas: nenhum
Gravado: data/processed/precedentes/precedentes_tst.jsonl · 1493 registros · 1022 buscáveis · sha256 2884779dbbd4cf4b…

Um registro do JSONL (Súmula 60):
{
 "id": "TST-SUM-60",
 "tipo": "sumula",
 "orgao": "Tribunal Pleno",
 "serie": "SUM",
 "numero": 60,
 "nome_citacao": "Súmula 60 do TST",
 "titulo": "ADICIONAL NOTURNO. INTEGRAÇÃO NO SALÁRIO E PRORROGAÇÃO EM HORÁRIO DIURNO",
 "texto": "I - O adicional noturno, pago com habitualidade, integra o salário do empregado para todos os efeitos. (ex-Súmula nº 60 - RA 105/1974, DJ 24.10.1974) II - Cumprida integralmente a jornada no período noturno e prorrogada esta, devido é também o adicional quanto às horas prorrogadas. Exegese do art. 73, § 5º, da CLT. (ex-OJ nº 6 da SBDI-1 - inserida em 25.11.1996)",
 "texto_busca": "I - O adicional noturno, pago com habitualidade, integra o salário do empregado para todos os efeitos. II - Cumprida integralmente a jornada no período noturno e prorrogada esta, devido é também o adicional 

,tipo,situacao,registros
0,orientacao_jurisprudencial,cancelada,308
1,orientacao_jurisprudencial,vigente,401
2,sumula,cancelada,138
3,sumula,suspensa,2
4,sumula,vigente,323
5,tema_repetitivo,afetado_sem_tese,98
6,tema_repetitivo,tese_fixada,221
7,tema_repetitivo,tese_superada,2


## 5. Texto oficial das súmulas do teste de posição

O teste de posição do notebook 00 usou textos de súmula digitados de memória, porque os documentos ainda não tinham chegado. Abaixo está o texto oficial, já estruturado, dos mesmos itens. O notebook 06 repete o teste com esse texto, dentro da base separada.

In [6]:
def item(rid, it=None):
    r = por_id[rid]
    if it:
        return next(i['texto'] for i in r['itens'] if i['item'] == it)
    return r['texto_busca']
for rid, it in [('TST-SUM-338', 'I'), ('TST-SUM-276', None), ('TST-SUM-378', 'II'), ('TST-SUM-6', 'VIII'), ('TST-SUM-60', None)]:
    print(f"{por_id[rid]['nome_citacao']}{', item ' + it if it else ''}: {est.sem_origem(item(rid, it))}\n")

Súmula 338 do TST, item I: É ônus do empregador que conta com mais de 10 (dez) empregados o registro da jornada de trabalho na forma do art. 74, § 2º, da CLT. A não-apresentação injustificada dos controles de frequência gera presunção relativa de veracidade da jornada de trabalho, a qual pode ser elidida por prova em contrário.

Súmula 276 do TST: O direito ao aviso prévio é irrenunciável pelo empregado. O pedido de dispensa de cumprimento não exime o empregador de pagar o respectivo valor, salvo comprovação de haver o prestador dos serviços obtido novo emprego.

Súmula 378 do TST, item II: São pressupostos para a concessão da estabilidade o afastamento superior a 15 dias e a conseqüente percepção do auxílio-doença acidentário, salvo se constatada, após a despedida, doença profissional que guarde relação de causalidade com a execução do contrato de emprego.

Súmula 6 do TST, item VIII: É do empregador o ônus da prova do fato impeditivo, modificativo ou extintivo da equiparação salarial

## Conclusões

- **JSONL único gravado:** súmulas, OJs e temas num só arquivo, com o mesmo esquema. Cada registro tem o nome de citação ("Súmula 60 do TST", "Tema 128 de IRR do TST"), que o prompt vai usar para identificar a fonte.
- **Só vai para a busca o que vale:**
  - os cancelados, as súmulas suspensas pelo STF, as teses superadas e os cancelamentos integrais noticiados ficam registrados, fora da busca;
  - o texto de busca não leva o histórico nem as notas.
- **Pendências:**
  1. **Livro oficial do TST de 2026**, ou a resolução dos cancelamentos, para confirmar a quarentena.
  2. **EC 141/2026** na base de legislação (art. 7º da Constituição).
  3. **Precedentes do STF**, a partir do índice de repercussão geral já recebido, numa etapa posterior.

Próximo: `04_chunking_precedentes.ipynb`, com um trecho por enunciado ou por item e o título repetido em cada item.